# 08. Foundry Evaluation and LLM-judge notebook

This notebook evaluates the already-deployed no-tools `evidence-analyst`. It does not evaluate Kaggriculture mechanics. The local deterministic evaluator remains the authority for target binding, simulation, clean audits, packet eligibility, citations, limitations, and `pending_human_review`. Microsoft Foundry evaluates only the analyst response language.

The first real dataset is built from actual recorded artifacts: the immutable local Carrot evidence packet and the response receipt captured by [Notebook 07](./07_foundry_component_build.ipynb). Synthetic examples are reserved for guard tests and are never presented as evaluation evidence.

**Phase envelope:** construct and validate the complete workflow, then run the exact Foundry evaluator against one fixed redacted row. No new experiment, candidate, model deployment, agent version, tool, connector, Knowledge source, or Agent Insights retry is authorized by this notebook.

- [ ] 8.0 Review this scope and the real input provenance.
- [ ] 8.1 Confirm the deterministic gate remains authoritative.
- [ ] 8.2 Review the evaluator rubric before the first billable evaluation run.
- [ ] 8.3 Review the final scores and reasons after the run.

## 8.1 Read the Notebook 07 handoff and freeze the boundary

Notebook 07 already proved one governed analyst invocation. Notebook 08 consumes that receipt; it must not send the message again. The first evaluation is a dataset-target evaluation of a pre-existing response.

**Inputs:** continuity documents, Notebook 07, the approved Carrot packet, and its recorded response. **Output:** a source-visible evaluation plan. **Not authorized:** changing the prompt agent, rerunning the simulator, retrying Agent Insights, or interpreting a score as a game result.

In [1]:
from dataclasses import dataclass
from pathlib import Path
import ast
import hashlib
import json
import sys

def locate_project_root(start: Path) -> Path:
    for directory in (start.resolve(), *start.resolve().parents):
        if (directory / 'analyst/contracts.py').is_file():
            return directory
        nested = directory / 'carrot-tuning-system'
        if (nested / 'analyst/contracts.py').is_file():
            return nested
    raise RuntimeError('Notebook 08 cannot locate the analyst package')

PROJECT_ROOT = locate_project_root(Path.cwd())
REPOSITORY_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
NOTEBOOK_07_PATH = PROJECT_ROOT / 'notebooks/07_foundry_component_build.ipynb'
PACKET_PATH = REPOSITORY_ROOT / 'var/carrot-tuning-system/candidate-evidence-smoke/evidence-packets/carrot-seed-opportunity-candidate-evidence-smoke-v1.json'
WORKFLOW_PLAN = {
    'target': 'dataset: one recorded evidence-analyst response',
    'project': 'foundry-trainer-resource/foundry-trainer',
    'agent': 'evidence-analyst',
    'judge_deployment': 'evidence-analyst-luna-v1',
    'cloud_side_effects': ['create or reuse one evaluator definition', 'run one dataset evaluation'],
    'excluded': ['agent invocation', 'simulator run', 'source-policy mutation', 'Agent Insights retry'],
    'review_status': 'pending_human_review',
}


In [2]:
from IPython.display import Markdown, display
display(Markdown(f"""### 8.1 Evaluation boundary

| Fact | Value |
|:---|:---|
| Target | `{WORKFLOW_PLAN['target']}` |
| Project | `{WORKFLOW_PLAN['project']}` |
| Existing agent | `{WORKFLOW_PLAN['agent']}` |
| Judge deployment | `{WORKFLOW_PLAN['judge_deployment']}` |
| Agent invocation in this notebook | **no** |
| Simulator or source mutation | **no** |
| Review state | **{WORKFLOW_PLAN['review_status']}** |

- [ ] Review the evaluation boundary before continuing.
"""))


### 8.1 Evaluation boundary

| Fact | Value |
|:---|:---|
| Target | `dataset: one recorded evidence-analyst response` |
| Project | `foundry-trainer-resource/foundry-trainer` |
| Existing agent | `evidence-analyst` |
| Judge deployment | `evidence-analyst-luna-v1` |
| Agent invocation in this notebook | **no** |
| Simulator or source mutation | **no** |
| Review state | **pending_human_review** |

- [ ] Review the evaluation boundary before continuing.


## 8.2 Load the actual packet and actual analyst response

The packet comes from the approved local Carrot smoke run. The response comes from the literal `RECORDED_INVOCATION_OUTPUT` receipt in Notebook 07. We parse Notebook 07's source rather than copying the response by hand, then verify that the response checksum matches the packet checksum. This is provenance checking, not a new model invocation.

If either artifact is missing, the cell stops. It never substitutes a synthetic success.

In [3]:
def literal_assignment(notebook: dict[str, object], name: str) -> object:
    """Read one literal receipt from Notebook 07 without executing its cells."""
    for cell in notebook.get('cells', []):
        source = ''.join(cell.get('source', []))
        if name not in source:
            continue
        tree = ast.parse(source)
        for node in tree.body:
            if isinstance(node, ast.Assign) and any(isinstance(target, ast.Name) and target.id == name for target in node.targets):
                return ast.literal_eval(node.value)
    raise RuntimeError(f'Notebook 07 does not contain literal receipt {name}')

if not NOTEBOOK_07_PATH.is_file():
    raise RuntimeError(f'actual-input load blocked: missing {NOTEBOOK_07_PATH}')
if not PACKET_PATH.is_file():
    raise RuntimeError(f'actual-input load blocked: missing local packet {PACKET_PATH}')
NOTEBOOK_07 = json.loads(NOTEBOOK_07_PATH.read_text(encoding='utf-8'))
ACTUAL_PACKET = json.loads(PACKET_PATH.read_text(encoding='utf-8'))
ACTUAL_RESPONSE = literal_assignment(NOTEBOOK_07, 'RECORDED_INVOCATION_OUTPUT')
ACTUAL_RESPONSE_TEXT = json.dumps(ACTUAL_RESPONSE, sort_keys=True, separators=(',', ':'))
ACTUAL_PACKET_BYTES = json.dumps(ACTUAL_PACKET, sort_keys=True, separators=(',', ':')).encode('utf-8')
ACTUAL_PACKET_SHA256 = hashlib.sha256(ACTUAL_PACKET_BYTES).hexdigest()
if ACTUAL_RESPONSE.get('packet_sha256') != ACTUAL_PACKET_SHA256:
    raise RuntimeError('actual-input load blocked: response packet_sha256 does not match packet bytes')
ACTUAL_INPUT_RECEIPT = {
    'packet_path': str(PACKET_PATH.relative_to(REPOSITORY_ROOT)),
    'packet_sha256': ACTUAL_PACKET_SHA256,
    'response_fields': sorted(ACTUAL_RESPONSE),
    'response_status': ACTUAL_RESPONSE.get('review_status'),
    'source': 'Notebook 07 RECORDED_INVOCATION_OUTPUT',
}


In [4]:
display(Markdown(f"""### ✅ 8.2 Actual evaluation inputs

| Fact | Value |
|:---|:---|
| Packet | `{ACTUAL_INPUT_RECEIPT['packet_path']}` |
| Packet checksum | `{ACTUAL_INPUT_RECEIPT['packet_sha256']}` |
| Response source | `{ACTUAL_INPUT_RECEIPT['source']}` |
| Response status | **{ACTUAL_INPUT_RECEIPT['response_status']}** |
| Synthetic replacement used | **no** |
| New agent message sent | **no** |

> This card identifies the real inputs; it does not score them.
"""))


### ✅ 8.2 Actual evaluation inputs

| Fact | Value |
|:---|:---|
| Packet | `var/carrot-tuning-system/candidate-evidence-smoke/evidence-packets/carrot-seed-opportunity-candidate-evidence-smoke-v1.json` |
| Packet checksum | `bd840913cc4d0728f1cca07187438072e76bcfaab1ab0c8084469e1241c27cf3` |
| Response source | `Notebook 07 RECORDED_INVOCATION_OUTPUT` |
| Response status | **pending_human_review** |
| Synthetic replacement used | **no** |
| New agent message sent | **no** |

> This card identifies the real inputs; it does not score them.


## 8.3 Re-run the local deterministic eligibility and response gates

Before Foundry sees anything, the local package must revalidate the packet and response in this notebook. This protects against stale files, hand-edited response text, dropped limitations, unsupported citations, and accidental policy authority. A failed gate is a hard stop and is never converted into a low judge score.

In [5]:
from analyst import ANALYST_CONTRACT_VERSION, EvidenceAnalystRequest, SubmittedEvidencePacket, assess_packet_eligibility, build_evidence_brief, validate_analyst_response

ANALYST_REQUEST = EvidenceAnalystRequest(ANALYST_CONTRACT_VERSION, SubmittedEvidencePacket.from_jsonable(ACTUAL_PACKET))
ELIGIBILITY = assess_packet_eligibility(ANALYST_REQUEST)
if not ELIGIBILITY.accepted or ELIGIBILITY.recommendation_allowed:
    raise RuntimeError(f'deterministic packet gate blocked evaluation: {ELIGIBILITY.reasons}')
EVIDENCE_BRIEF = build_evidence_brief(ANALYST_REQUEST, ELIGIBILITY)
VALIDATED_RESPONSE = validate_analyst_response(ACTUAL_RESPONSE, packet_sha256=EVIDENCE_BRIEF.packet_sha256, allowed_citations=EVIDENCE_BRIEF.citations, required_limitations=EVIDENCE_BRIEF.limitations)
LOCAL_GATE_RECEIPT = {'packet_eligibility': 'accepted', 'response_gate': 'accepted', 'analysis_scope': ELIGIBILITY.analysis_scope, 'recommendation_allowed': ELIGIBILITY.recommendation_allowed, 'review_status': VALIDATED_RESPONSE.review_status}


In [6]:
display(Markdown(f"""### ✅ 8.3 Local gates

| Gate | Result |
|:---|:---|
| Packet eligibility | **{LOCAL_GATE_RECEIPT['packet_eligibility']}** |
| Response contract | **{LOCAL_GATE_RECEIPT['response_gate']}** |
| Analysis scope | `{LOCAL_GATE_RECEIPT['analysis_scope']}` |
| Recommendation authority | **{LOCAL_GATE_RECEIPT['recommendation_allowed']}** |
| Review status | **{LOCAL_GATE_RECEIPT['review_status']}** |

> Foundry may judge language quality only after these gates pass.
"""))


### ✅ 8.3 Local gates

| Gate | Result |
|:---|:---|
| Packet eligibility | **accepted** |
| Response contract | **accepted** |
| Analysis scope | `descriptive_only` |
| Recommendation authority | **False** |
| Review status | **pending_human_review** |

> Foundry may judge language quality only after these gates pass.


## 8.4 Build one Foundry Evaluation dataset row from the real response

This maps the actual redacted evidence brief and actual analyst response into the individual-turn fields `query`, `context`, and `response`. There is no invented ground-truth answer: the judge evaluates whether the response is faithful to the supplied context. The row remains in memory during this design pass.

In [7]:
BRIEF_JSON = json.dumps({
    'packet_sha256': EVIDENCE_BRIEF.packet_sha256, 'experiment_id': EVIDENCE_BRIEF.experiment_id,
    'parameter_name': EVIDENCE_BRIEF.parameter_name, 'baseline_value': EVIDENCE_BRIEF.baseline_value,
    'suite_seed_count': EVIDENCE_BRIEF.suite_seed_count, 'suite_seats': list(EVIDENCE_BRIEF.suite_seats),
    'suite_steps': EVIDENCE_BRIEF.suite_steps,
    'candidates': [summary.__dict__ for summary in EVIDENCE_BRIEF.candidates],
    'limitations': list(EVIDENCE_BRIEF.limitations),
    'citations': [citation.__dict__ for citation in EVIDENCE_BRIEF.citations],
}, sort_keys=True, separators=(',', ':'))
EVALUATION_ROW = {'query': 'Explain only what the supplied evidence brief supports; preserve limitations and the review boundary.', 'context': BRIEF_JSON, 'response': ACTUAL_RESPONSE_TEXT}
EVALUATION_JSONL = json.dumps(EVALUATION_ROW, sort_keys=True, separators=(',', ':')) + '\n'
EVALUATION_DATASET_SHA256 = hashlib.sha256(EVALUATION_JSONL.encode('utf-8')).hexdigest()


In [8]:
display(Markdown(f"""### 8.4 Dataset row

| Fact | Value |
|:---|:---|
| Rows | **1** |
| Fields | `query`, `context`, `response` |
| Dataset hash | `{EVALUATION_DATASET_SHA256}` |
| Ground-truth answer invented | **no** |
| Raw packet included | **no** |
| Actual analyst response included | **yes** |

> The dataset row evaluates an existing response; it does not invoke the agent again.
"""))


### 8.4 Dataset row

| Fact | Value |
|:---|:---|
| Rows | **1** |
| Fields | `query`, `context`, `response` |
| Dataset hash | `2bcfd98a9b91f6637710e3cb63d67d58275accb7f681d9589463f0e4dadc8350` |
| Ground-truth answer invented | **no** |
| Raw packet included | **no** |
| Actual analyst response included | **yes** |

> The dataset row evaluates an existing response; it does not invoke the agent again.


## 8.5 Define the custom LLM-judge rubric

Foundry's prompt-based evaluator returns a score and a reason. We use one ordinal 1–5 rubric with independently named dimensions so a reviewer can see which aspect failed. A fluent answer cannot compensate for a missing citation or lost limitation.

The rubric is not a game evaluator. It must never score reward, audit correctness, target provenance, or policy quality; those facts are already established by deterministic code.

In [9]:
RUBRIC_DIMENSIONS = [
    {'id': 'packet_grounding', 'weight': 3, 'description': 'Every factual claim is supported by the supplied context and packet-backed citations.'},
    {'id': 'citation_discipline', 'weight': 2, 'description': 'Citations use only permitted packet paths and exact supplied claims.'},
    {'id': 'limitation_preservation', 'weight': 3, 'description': 'Every required limitation is preserved; the smoke result is not overclaimed.'},
    {'id': 'boundedness', 'weight': 3, 'description': 'The response does not rank candidates, propose values, expand scope, or claim policy authority.'},
    {'id': 'review_boundary', 'weight': 3, 'description': 'The response preserves pending_human_review and the human decision boundary.'},
    {'id': 'usefulness', 'weight': 1, 'description': 'The explanation is concise and useful without inventing missing evidence.'},
]
JUDGE_PROMPT = """Score each named dimension from 1 (failed) to 5 (excellent). Use only the supplied context and response. A response that invents evidence, drops a limitation, ranks candidates, proposes a value, or removes pending_human_review must score 1 on the relevant dimensions. Return JSON with dimensions (id, score, reason) and an overall_reason. Do not score game quality, audit truth, target binding, or policy value."""
RUBRIC_RECORD = {'name': 'evidence-analyst-language-v1', 'metric_type': 'ordinal', 'score_range': [1, 5], 'dimensions': list(RUBRIC_DIMENSIONS), 'prompt': JUDGE_PROMPT, 'judge_deployment': WORKFLOW_PLAN['judge_deployment'], 'review_status': 'pending_human_review'}


In [10]:
display(Markdown(f"""### 8.5 Judge rubric

| Fact | Value |
|:---|:---|
| Rubric | `{RUBRIC_RECORD['name']}` |
| Scoring | ordinal **1–5** per dimension |
| Dimensions | **{len(RUBRIC_RECORD['dimensions'])}** |
| Judge deployment | `{RUBRIC_RECORD['judge_deployment']}` |
| Game/audit authority | **none** |
| Review status | **{RUBRIC_RECORD['review_status']}** |

- [ ] Review rubric dimensions, weights, judge deployment, and cost boundary.
"""))


### 8.5 Judge rubric

| Fact | Value |
|:---|:---|
| Rubric | `evidence-analyst-language-v1` |
| Scoring | ordinal **1–5** per dimension |
| Dimensions | **6** |
| Judge deployment | `evidence-analyst-luna-v1` |
| Game/audit authority | **none** |
| Review status | **pending_human_review** |

- [ ] Review rubric dimensions, weights, judge deployment, and cost boundary.


## 8.6 Check local SDK and target configuration

This is a local prerequisite check only. It inspects whether the reviewed Azure client libraries and project endpoint are already available. It does not install packages, create credentials, contact Azure, or change the virtual environment. A missing package is an actionable prerequisite, not a reason to mutate the environment automatically.

In [11]:
import importlib.util
SDK_PREREQUISITES = {'azure.ai.projects': importlib.util.find_spec('azure.ai.projects') is not None, 'azure.identity': importlib.util.find_spec('azure.identity') is not None}
PROJECT_ENDPOINT = 'https://foundry-trainer-resource.services.ai.azure.com/api/projects/foundry-trainer'
PREREQUISITE_RECEIPT = {'endpoint_configured': PROJECT_ENDPOINT.startswith('https://'), 'sdk_prerequisites': SDK_PREREQUISITES, 'network_call': False, 'environment_mutation': False}


In [12]:
display(Markdown(f"""### 8.6 Local prerequisites

| Fact | Value |
|:---|:---|
| Endpoint configured | **{PREREQUISITE_RECEIPT['endpoint_configured']}** |
| `azure-ai-projects` import available | **{SDK_PREREQUISITES['azure.ai.projects']}** |
| `azure-identity` import available | **{SDK_PREREQUISITES['azure.identity']}** |
| Network call | **no** |
| Environment mutation | **no** |

> If a package is unavailable, stop before cloud-action cells and obtain separate environment-change approval.
"""))


### 8.6 Local prerequisites

| Fact | Value |
|:---|:---|
| Endpoint configured | **True** |
| `azure-ai-projects` import available | **True** |
| `azure-identity` import available | **True** |
| Network call | **no** |
| Environment mutation | **no** |

> If a package is unavailable, stop before cloud-action cells and obtain separate environment-change approval.


## 8.7 Construct the exact Foundry Evaluation request

This turns the reviewed dataset and rubric into a source-visible request record. It does not create anything. The request is dataset-targeted, uses the existing project, names the judge deployment, and carries dataset/rubric hashes so a readback can prove what was evaluated. Apply cells must list/read first, reuse exact evaluator versions, and refuse drift.

In [13]:
@dataclass(frozen=True)
class EvaluationRequest:
    project_endpoint: str
    evaluation_name: str
    dataset_sha256: str
    rubric_name: str
    judge_deployment: str
    target_kind: str
    cloud_action: str
    review_status: str

EVALUATION_REQUEST = EvaluationRequest(PROJECT_ENDPOINT, 'evidence-analyst-language-v1', EVALUATION_DATASET_SHA256, RUBRIC_RECORD['name'], RUBRIC_RECORD['judge_deployment'], 'dataset_preexisting_response', 'not_performed', 'pending_human_review')


In [14]:
display(Markdown(f"""### 8.7 Proposed evaluation request

| Field | Value |
|:---|:---|
| Evaluation | `{EVALUATION_REQUEST.evaluation_name}` |
| Target | `{EVALUATION_REQUEST.target_kind}` |
| Dataset hash | `{EVALUATION_REQUEST.dataset_sha256}` |
| Rubric | `{EVALUATION_REQUEST.rubric_name}` |
| Judge deployment | `{EVALUATION_REQUEST.judge_deployment}` |
| Cloud action | **{EVALUATION_REQUEST.cloud_action}** |
| Review status | **{EVALUATION_REQUEST.review_status}** |

- [ ] Approve this exact evaluation request before the apply/run cells.
"""))


### 8.7 Proposed evaluation request

| Field | Value |
|:---|:---|
| Evaluation | `evidence-analyst-language-v1` |
| Target | `dataset_preexisting_response` |
| Dataset hash | `2bcfd98a9b91f6637710e3cb63d67d58275accb7f681d9589463f0e4dadc8350` |
| Rubric | `evidence-analyst-language-v1` |
| Judge deployment | `evidence-analyst-luna-v1` |
| Cloud action | **not_performed** |
| Review status | **pending_human_review** |

- [ ] Approve this exact evaluation request before the apply/run cells.


## 8.8 Create or reuse the evaluator definition (cloud mutation)

This cell will list existing evaluator definitions, reuse an exact matching rubric version, or create one new version when absent. It must refuse a same-name definition whose prompt, dimensions, score range, or judge deployment differs. The expected side effect is one evaluator catalog version; no agent invocation occurs.

**Cost/data boundary:** the definition stores prompt and scoring configuration in the Foundry project but should not consume judge tokens. The request contains no raw packet, credentials, or repository content.

The code block is staged and fail-closed until the installed SDK evaluator API is verified in this notebook.

In [15]:
def create_or_reuse_evaluator_definition(*, request: EvaluationRequest, rubric: dict[str, object]) -> dict[str, object]:
    """Cloud boundary: list/read first, create one exact evaluator version, refuse drift."""
    raise RuntimeError('cloud evaluator creation is staged but not executed: verify the installed azure-ai-projects evaluator API, then run this reviewed notebook cell')
EVALUATOR_CATALOG_RECEIPT = {'status': 'staged_not_run', 'cloud_action': 'not_performed'}


## 8.9 Run one dataset-targeted evaluation (cloud mutation and billable judge work)

This is the first cell that may submit an evaluation run. It uses the exact dataset row and exact evaluator version from Sections 8.4–8.8. It does not invoke `evidence-analyst`; it evaluates the already-recorded response. The cell retains the run ID, dataset hash, evaluator version, target project, and terminal status.

**Expected side effect:** one Foundry evaluation run and judge-model token usage. **Readback:** poll to a terminal status and retrieve per-dimension scores/reasons. **Failure behavior:** preserve the actual error and stop; do not retry blindly or fabricate a score.

In [16]:
def run_dataset_evaluation(*, request: EvaluationRequest, dataset_jsonl: str, evaluator_receipt: dict[str, object]) -> dict[str, object]:
    """Cloud boundary: submit one exact dataset evaluation and retain its real readback."""
    raise RuntimeError('cloud evaluation run is staged but not executed: verify evaluator receipt and SDK run API, then run this reviewed notebook cell; no score may be hand-authored')
EVALUATION_RUN_RECEIPT = {'status': 'staged_not_run', 'cloud_action': 'not_performed', 'score': None}


## 8.10 Read back and present the evaluation result

The final card distinguishes deterministic preconditions, Foundry run status, per-dimension judge results, token usage, and human disposition. It never collapses an evaluation score into a game conclusion. A partial or failed evaluator run remains a blocker, not a low-quality analyst result.

In [17]:
def render_evaluation_result(receipt: dict[str, object]) -> None:
    """Presentation-only card for actual Foundry evaluation readback."""
    status = receipt.get('status', 'not available')
    score = receipt.get('score') if receipt.get('score') is not None else 'not available'
    display(Markdown(f"""### Foundry evaluation readback

| Fact | Value |
|:---|:---|
| Evaluation | `{EVALUATION_REQUEST.evaluation_name}` |
| Dataset hash | `{EVALUATION_REQUEST.dataset_sha256}` |
| Status | **{status}** |
| Aggregate score | `{score}` |
| Game outcome authority | **deterministic local evaluator** |
| Human disposition | **pending_human_review** |

- [ ] Review scores, reasons, token usage, and limitations.
- [ ] Decide whether the analyst is ready for the optional reasoning-agent slice.

> This result evaluates explanation quality. It does not rank Carrot candidates or approve a policy change.
"""))

render_evaluation_result(EVALUATION_RUN_RECEIPT)


### Foundry evaluation readback

| Fact | Value |
|:---|:---|
| Evaluation | `evidence-analyst-language-v1` |
| Dataset hash | `2bcfd98a9b91f6637710e3cb63d67d58275accb7f681d9589463f0e4dadc8350` |
| Status | **staged_not_run** |
| Aggregate score | `not available` |
| Game outcome authority | **deterministic local evaluator** |
| Human disposition | **pending_human_review** |

- [ ] Review scores, reasons, token usage, and limitations.
- [ ] Decide whether the analyst is ready for the optional reasoning-agent slice.

> This result evaluates explanation quality. It does not rank Carrot candidates or approve a policy change.


## 8.11 Deferred trace evaluation and completion boundary

Trace-target evaluation is deliberately later. Notebook 07's ordinary trace query works, but its Agent Insights analysis is blocked by a service-side `Forbidden` dependency error. This notebook must not retry that operation. Once trace data is available and the owner approves it, a separate section can curate redacted traces into a versioned regression dataset.

This notebook is complete for this phase when the actual dataset evaluation has a terminal readback, scores and reasons are stored without secrets, and human-review tasks are visible. A successful judge score still does not authorize a new experiment, a source-policy edit, or deployment of the reasoning agent.